# Student-selected LightGBM training-length comparison

**Decision recorded on 11 October 2026:** the student chose to train LightGBM with **150, 225 and 300 trees**, keeping all other settings fixed. The comparison uses the same 15 chronological checks as the current submitted XGBoost/LightGBM ensemble, whose reported public ROC-AUC is **0.94701**.

Each tree makes another correction to the model's predictions. More corrections can capture useful relationships, but can also fit patterns specific to old transactions. This experiment tests training length, rather than changing multiple settings together. All three candidates are declared before training; there is no automatic search or selection. Current ensemble controls are freshly fitted on matching rows.

The 15 checks train through steps 7–21, respectively, and predict the next 14 steps. Results also show the seven later checks, the four previously chosen checks (cutoffs 14, 17, 19, 21), and the latest check. These reused, overlapping checks are development evidence; their average is not an estimate of the hidden Kaggle score.

Only confirmed labels fit these supervised models. Unknown training rows are counted but do not receive invented labels. Earlier student-selected semi-supervised experiments did not improve validation; this chosen experiment returns to the strongest recorded boosting approach. Future validation and test rows never fit a model, feature filter or preprocessing step. All original 165 features enter LightGBM; the existing ensemble's XGBoost feature filter remains training-only.

**Assistance disclosure:** the student chose the fixed experiment and validation design. OpenAI Codex implemented and ran it, checked artifacts and drafted the factual record. The student retains responsibility for explaining the choices, interpreting results and choosing submissions. No Kaggle upload or guaranteed improvement is part of this notebook.

Run all cells in a fresh CPU Python kernel. Set `DATA_DIR` and `OUTPUT_DIR` below if necessary. Prediction generation uses only this notebook, the supplied competition files and standard recorded libraries; prior prediction files are not required.


In [1]:
from pathlib import Path
import os, json, hashlib, time, platform
for variable in ("OMP_NUM_THREADS", "OPENBLAS_NUM_THREADS", "MKL_NUM_THREADS"):
    os.environ[variable] = "4"
import numpy as np
import pandas as pd
import sklearn, xgboost, lightgbm
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier, Booster
from sklearn.metrics import roc_auc_score
from scipy.stats import spearmanr
from threadpoolctl import threadpool_limits

DATA_DIR = Path(os.environ.get("BT4012_DATA_DIR", "/Users/mlvlo/BT4012/Kaggle/Data Set"))
if not DATA_DIR.exists():
    DATA_DIR = Path("/kaggle/input") if Path("/kaggle/input").exists() else Path.cwd()
OUTPUT_DIR = Path(os.environ.get("BT4012_LENGTH_OUTPUT_DIR", str(Path.cwd() / "lightgbm_length_results")))
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
FEATURES = [f"feat_{i}" for i in range(1, 166)]
TREE_COUNTS = (150, 225, 300)
CHECKS = tuple((end, end + 1, end + 14) for end in range(7, 22))
SELECTED_FOUR = (14, 17, 19, 21)
SEED = 42
STARTED = time.perf_counter()
FIXED_LGBM = dict(learning_rate=0.05, num_leaves=15, max_bin=63,
                 min_child_samples=5, min_child_weight=0.001, reg_lambda=1,
                 objective="binary", deterministic=True, force_col_wise=True,
                 verbosity=-1, n_jobs=4, random_state=SEED)
RECIPE = dict(tree_counts=list(TREE_COUNTS), fixed_lightgbm=FIXED_LGBM,
              chronological_checks=[list(check) for check in CHECKS],
              selected_four=list(SELECTED_FOUR), existing_public_auc=0.94701,
              student_decision="Train the fixed LightGBM comparison (Recommended)",
              final_selection="pending student review", kaggle_upload=False)
recipe_file = OUTPUT_DIR / "declared_recipe.json"
if recipe_file.exists():
    assert json.loads(recipe_file.read_text()) == RECIPE, "Different prior recipe: use a new output folder."
else:
    recipe_file.write_text(json.dumps(RECIPE, indent=2))

def digest(path):
    return hashlib.sha256(Path(path).read_bytes()).hexdigest()

def provided_file(name):
    matches = sorted(DATA_DIR.rglob(name))
    assert len(matches) == 1, f"Expected exactly one provided {name}: found {matches}"
    return matches[0]

def progress(phase, **details):
    (OUTPUT_DIR / "progress.json").write_text(json.dumps(dict(
        phase=phase, elapsed_seconds=round(time.perf_counter() - STARTED, 3), **details), indent=2))

print("Declared candidates:", TREE_COUNTS, "trees; fixed settings:", FIXED_LGBM)
print("Results folder:", OUTPUT_DIR)

Declared candidates: (150, 225, 300) trees; fixed settings: {'learning_rate': 0.05, 'num_leaves': 15, 'max_bin': 63, 'min_child_samples': 5, 'min_child_weight': 0.001, 'reg_lambda': 1, 'objective': 'binary', 'deterministic': True, 'force_col_wise': True, 'verbosity': -1, 'n_jobs': 4, 'random_state': 42}
Results folder: /Users/mlvlo/.codex/.chatgpt-projects/g-p-6a901a23216481918fe5444e40decc24/student_model_review/lightgbm_length_results


In [2]:
train_path = provided_file("Train Data.csv")
test_path = provided_file("Test Data.csv")
sample_path = provided_file("BT4012 Competition Submission.csv")
SOURCE_HASHES = {str(p): digest(p) for p in (train_path, test_path, sample_path)}
train = pd.read_csv(train_path)
numeric_labels = pd.to_numeric(train["label"], errors="coerce")
known = train.loc[numeric_labels.notna()].copy()
known["label"] = numeric_labels.loc[numeric_labels.notna()].astype(int)
test = pd.read_csv(test_path)
sample = pd.read_csv(sample_path)
assert numeric_labels.dropna().isin([0, 1]).all()
assert known["label"].nunique() == 2
assert train["txId"].is_unique and test["txId"].is_unique
assert set(train["txId"]).isdisjoint(test["txId"])
assert train["time_step"].max() < test["time_step"].min()
assert test["index"].is_unique and sample["index"].is_unique
assert set(test["index"]) == set(sample["index"])
assert sample.columns.tolist() == ["index", "target"]
for rows in (train, test):
    assert np.isfinite(rows[FEATURES].to_numpy()).all()

def make_lightgbm(trees):
    assert trees in TREE_COUNTS
    return LGBMClassifier(n_estimators=trees, **FIXED_LGBM)

def existing_xgb_columns(learning):
    # Preserve the submitted ensemble's filter, using fitting data only.
    cutoff = int(learning["time_step"].max())
    filter_rows = learning.loc[learning["time_step"] <= cutoff - 3]
    correlations = filter_rows[FEATURES].corrwith(filter_rows["time_step"], method="spearman")
    removed = correlations.index[correlations.abs() > 0.9].tolist()
    return [column for column in FEATURES if column not in removed], removed

def make_existing_xgb():
    return XGBClassifier(n_estimators=200, learning_rate=0.05, tree_method="hist",
                         grow_policy="lossguide", max_depth=0, max_leaves=15,
                         max_bin=127, min_child_weight=1, reg_lambda=1,
                         objective="binary:logistic", n_jobs=4, random_state=SEED)

print(f"Training rows: {len(train):,}; confirmed labels: {len(known):,}; "
      f"unknown rows: {int(numeric_labels.isna().sum()):,}; test rows: {len(test):,}")
print("Confirmed labels:", known["label"].value_counts().to_dict())

Training rows: 141,772; confirmed labels: 31,235; unknown rows: 110,537; test rows: 15,329
Confirmed labels: {0: 27591, 1: 3644}


In [3]:
# Fixed outer evaluations. No early stopping or parameter choice uses checking labels.
results_path = OUTPUT_DIR / "validation_results.csv"
arrays_path = OUTPUT_DIR / "validation_probabilities.npz"
records, arrays, selections = [], {}, {}
if results_path.exists():
    assert arrays_path.exists()
    records = pd.read_csv(results_path).to_dict(orient="records")
    with np.load(arrays_path) as saved:
        arrays = {key: saved[key] for key in saved.files}
    assert len(records) == len({int(row["train_end"]) for row in records})
for cutoff, first, last in CHECKS:
    learning = known.loc[known["time_step"] <= cutoff]
    checking = known.loc[known["time_step"].between(first, last)]
    assert learning["time_step"].max() < checking["time_step"].min()
    assert set(learning["txId"]).isdisjoint(checking["txId"])
    assert learning["label"].nunique() == checking["label"].nunique() == 2
    columns, removed = existing_xgb_columns(learning)
    selections[str(cutoff)] = removed
    if any(int(row["train_end"]) == cutoff for row in records):
        assert np.array_equal(arrays[f"labels_{cutoff}"], checking["label"].to_numpy())
        assert np.array_equal(arrays[f"txIds_{cutoff}"], checking["txId"].to_numpy())
        continue
    progress("chronological training", cutoff=cutoff, completed_checks=len(records), total_checks=len(CHECKS))
    tick = time.perf_counter()
    scores = {}
    with threadpool_limits(limits=4):
        for trees in TREE_COUNTS:
            model = make_lightgbm(trees)
            model.fit(learning[FEATURES], learning["label"])
            scores[f"lightgbm_{trees}"] = model.predict_proba(checking[FEATURES])[:, 1]
            assert model.n_estimators_ == trees
        xgb_model = make_existing_xgb()
        xgb_model.fit(learning[columns], learning["label"])
        p_xgb = xgb_model.predict_proba(checking[columns])[:, 1]
    scores["current_pair"] = 0.5 * p_xgb + 0.5 * scores["lightgbm_150"]
    labels = checking["label"].to_numpy()
    row = dict(train_end=cutoff, validation_start=first, validation_end=last,
               learning_rows=len(learning), checking_rows=len(checking),
               learning_illicit=int(learning["label"].sum()), checking_illicit=int(labels.sum()),
               elapsed_seconds=round(time.perf_counter() - tick, 3))
    for name, probabilities in scores.items():
        assert len(probabilities) == len(checking)
        assert np.isfinite(probabilities).all() and ((probabilities >= 0) & (probabilities <= 1)).all()
        row[name + "_auc"] = float(roc_auc_score(labels, probabilities))
        arrays[f"{name}_{cutoff}"] = probabilities
    for trees in TREE_COUNTS:
        row[f"lightgbm_{trees}_minus_pair"] = row[f"lightgbm_{trees}_auc"] - row["current_pair_auc"]
        row[f"lightgbm_{trees}_minus_150"] = row[f"lightgbm_{trees}_auc"] - row["lightgbm_150_auc"]
    arrays[f"labels_{cutoff}"] = labels
    arrays[f"txIds_{cutoff}"] = checking["txId"].to_numpy()
    arrays[f"time_steps_{cutoff}"] = checking["time_step"].to_numpy()
    records.append(row)
    np.savez_compressed(arrays_path, **arrays)
    pd.DataFrame(records).to_csv(results_path, index=False)
    print(f"Cutoff {cutoff:2}: pair {row['current_pair_auc']:.6f}; "
          + "; ".join(f"LGB {trees}: {row[f'lightgbm_{trees}_auc']:.6f}" for trees in TREE_COUNTS))
validation = pd.DataFrame(records).sort_values("train_end").reset_index(drop=True)
assert validation["train_end"].tolist() == [check[0] for check in CHECKS]
validation.to_csv(results_path, index=False)
(OUTPUT_DIR / "xgboost_removed_features.json").write_text(json.dumps(selections, indent=2))
progress("all chronological checks complete", completed_checks=len(records))
display(validation[["train_end", "current_pair_auc"] + [f"lightgbm_{n}_auc" for n in TREE_COUNTS]])

Cutoff  7: pair 0.992985; LGB 150: 0.991691; LGB 225: 0.992252; LGB 300: 0.992588
Cutoff  8: pair 0.993827; LGB 150: 0.993200; LGB 225: 0.993783; LGB 300: 0.993993
Cutoff  9: pair 0.991915; LGB 150: 0.990934; LGB 225: 0.991591; LGB 300: 0.991521
Cutoff 10: pair 0.977749; LGB 150: 0.976330; LGB 225: 0.975135; LGB 300: 0.974008
Cutoff 11: pair 0.975593; LGB 150: 0.973871; LGB 225: 0.971075; LGB 300: 0.967252
Cutoff 12: pair 0.969104; LGB 150: 0.968787; LGB 225: 0.966885; LGB 300: 0.964336
Cutoff 13: pair 0.971758; LGB 150: 0.969207; LGB 225: 0.968999; LGB 300: 0.969035
Cutoff 14: pair 0.972177; LGB 150: 0.968360; LGB 225: 0.969240; LGB 300: 0.967026
Cutoff 15: pair 0.939024; LGB 150: 0.943194; LGB 225: 0.939887; LGB 300: 0.936142
Cutoff 16: pair 0.947183; LGB 150: 0.948616; LGB 225: 0.944501; LGB 300: 0.939395
Cutoff 17: pair 0.939401; LGB 150: 0.941784; LGB 225: 0.936758; LGB 300: 0.935291
Cutoff 18: pair 0.916818; LGB 150: 0.923267; LGB 225: 0.912162; LGB 300: 0.911618
Cutoff 19: pair 

,train_end,current_pair_auc,lightgbm_150_auc,lightgbm_225_auc,lightgbm_300_auc
0,7,0.992985,0.991691,0.992252,0.992588
1,8,0.993827,0.993200,0.993783,0.993993
2,9,0.991915,0.990934,0.991591,0.991521
3,10,0.977749,0.976330,0.975135,0.974008
4,11,0.975593,0.973871,0.971075,0.967252
5,12,0.969104,0.968787,0.966885,0.964336
6,13,0.971758,0.969207,0.968999,0.969035
7,14,0.972177,0.968360,0.969240,0.967026
8,15,0.939024,0.943194,0.939887,0.936142
9,16,0.947183,0.948616,0.944501,0.939395


In [4]:
GROUPS = {
    "all_15": validation,
    "earlier_eight": validation.loc[validation["train_end"] <= 14],
    "later_seven": validation.loc[validation["train_end"] >= 15],
    "selected_four": validation.loc[validation["train_end"].isin(SELECTED_FOUR)],
    "latest": validation.loc[validation["train_end"] == 21],
}
summary_rows = []
for group_name, rows in GROUPS.items():
    for name in ["current_pair"] + [f"lightgbm_{n}" for n in TREE_COUNTS]:
        values = rows[name + "_auc"]
        summary_rows.append(dict(group=group_name, model=name, checks=len(rows),
            mean_auc=float(values.mean()), worst_auc=float(values.min()),
            mean_delta_vs_pair=float((values - rows["current_pair_auc"]).mean()),
            wins_vs_pair=int((values > rows["current_pair_auc"]).sum()),
            mean_delta_vs_150=float((values - rows["lightgbm_150_auc"]).mean()),
            wins_vs_150=int((values > rows["lightgbm_150_auc"]).sum())))
summary = pd.DataFrame(summary_rows)
summary.to_csv(OUTPUT_DIR / "summary.csv", index=False)
per_period = []
for cutoff, _, _ in CHECKS:
    periods = arrays[f"time_steps_{cutoff}"]
    labels = arrays[f"labels_{cutoff}"]
    for step in np.unique(periods):
        mask = periods == step
        row = dict(train_end=cutoff, time_step=int(step), rows=int(mask.sum()), illicit=int(labels[mask].sum()))
        for name in ["current_pair"] + [f"lightgbm_{n}" for n in TREE_COUNTS]:
            row[name + "_auc"] = float(roc_auc_score(labels[mask], arrays[f"{name}_{cutoff}"][mask])) if len(np.unique(labels[mask])) == 2 else np.nan
        per_period.append(row)
pd.DataFrame(per_period).to_csv(OUTPUT_DIR / "per_time_step.csv", index=False)
checking_ids = np.concatenate([arrays[f"txIds_{end}"] for end, _, _ in CHECKS])
unique_ids, appearances = np.unique(checking_ids, return_counts=True)
display(summary)
print(f"Overlapping checks: {len(checking_ids):,} row appearances; {len(unique_ids):,} unique transactions; "
      f"up to {int(appearances.max())} appearances per transaction.")
print("Public ROC-AUC of these three standalone candidates is unknown. Model selection remains with the student.")

Overlapping checks: 163,794 row appearances; 21,682 unique transactions; up to 14 appearances per transaction.
Public ROC-AUC of these three standalone candidates is unknown. Model selection remains with the student.


,group,model,checks,mean_auc,worst_auc,mean_delta_vs_pair,wins_vs_pair,mean_delta_vs_150,wins_vs_150
0,all_15,current_pair,15,0.954972,0.907205,0.000000,0,-0.000761,8
1,all_15,lightgbm_150,15,0.955732,0.908792,0.000761,7,0.000000,0
2,all_15,lightgbm_225,15,0.952160,0.894119,-0.002811,1,-0.003572,4
3,all_15,lightgbm_300,15,0.950552,0.888088,-0.004420,1,-0.005180,3
4,earlier_eight,current_pair,8,0.980638,0.969104,0.000000,0,0.001591,8
5,earlier_eight,lightgbm_150,8,0.979047,0.968360,-0.001591,0,0.000000,0
6,earlier_eight,lightgbm_225,8,0.978620,0.966885,-0.002019,0,-0.000427,4
7,earlier_eight,lightgbm_300,8,0.977470,0.964336,-0.003168,1,-0.001577,3
8,later_seven,current_pair,7,0.925638,0.907205,0.000000,0,-0.003448,0
9,later_seven,lightgbm_150,7,0.929086,0.908792,0.003448,7,0.000000,0


In [5]:
# Fit all three declared candidates on all confirmed labels through step 35.
# Test features are used only to generate predictions after fitting.
final_rows, full_predictions = [], {}
for trees in TREE_COUNTS:
    progress("full training and artifact checks", trees=trees)
    model = make_lightgbm(trees)
    with threadpool_limits(limits=4):
        model.fit(known[FEATURES], known["label"])
        probabilities = model.predict_proba(test[FEATURES])[:, 1]
    assert model.n_estimators_ == trees
    aligned = sample["index"].map(pd.Series(probabilities, index=test["index"])).to_numpy()
    assert len(aligned) == len(sample) and np.isfinite(aligned).all()
    assert ((aligned >= 0) & (aligned <= 1)).all()
    model_path = OUTPUT_DIR / f"lightgbm_{trees}_model.txt"
    model.booster_.save_model(str(model_path))
    with threadpool_limits(limits=4):
        restored = Booster(model_file=str(model_path))
        restored_probabilities = restored.predict(test[FEATURES], num_threads=4)
    assert restored.feature_name() == FEATURES
    assert restored.num_trees() == trees
    reload_error = float(np.max(np.abs(restored_probabilities - probabilities)))
    assert reload_error <= 1e-15
    submission = pd.DataFrame({"index": sample["index"], "target": aligned})
    csv_path = OUTPUT_DIR / f"submission_lightgbm_{trees}_experimental.csv"
    submission.to_csv(csv_path, index=False)
    reread = pd.read_csv(csv_path)
    assert reread.columns.tolist() == ["index", "target"]
    assert reread["index"].equals(sample["index"])
    assert np.isfinite(reread["target"]).all() and reread["target"].between(0, 1).all()
    assert np.allclose(reread["target"], aligned, rtol=0, atol=1e-15)
    full_predictions[f"lightgbm_{trees}"] = aligned
    final_rows.append(dict(model=f"lightgbm_{trees}", trees=trees, training_rows=len(known),
        training_illicit=int(known["label"].sum()), test_rows=len(sample),
        unknown_training_rows_excluded=int(numeric_labels.isna().sum()),
        saved_model_reload_max_error=reload_error, submission_sha256=digest(csv_path),
        model_sha256=digest(model_path), submission=str(csv_path)))
    print(f"Saved and verified {trees}-tree model and {len(sample):,} prediction rows.")
full_predictions["indices"] = sample["index"].to_numpy()
np.savez_compressed(OUTPUT_DIR / "test_probabilities.npz", **full_predictions)
final_training = pd.DataFrame(final_rows)
final_training.to_csv(OUTPUT_DIR / "final_training.csv", index=False)
metadata = dict(RECIPE, source_sha256=SOURCE_HASHES,
    train_rows=len(train), known_training_rows=len(known), unknown_training_rows=int(numeric_labels.isna().sum()),
    test_rows=len(test), unique_checking_transactions=len(unique_ids),
    checking_row_appearances=len(checking_ids), maximum_checking_appearances=int(appearances.max()),
    future_rows_used_for_fitting=False, unknown_rows_given_labels=False,
    all_saved_models_verified=True, outer_auc_used_for_stopping=False,
    original_data_unchanged=all(digest(path) == value for path, value in SOURCE_HASHES.items()),
    python=platform.python_version(), numpy=np.__version__, pandas=pd.__version__,
    sklearn=sklearn.__version__, lightgbm=lightgbm.__version__, xgboost=xgboost.__version__,
    elapsed_seconds=round(time.perf_counter() - STARTED, 3))
assert metadata["original_data_unchanged"]
(OUTPUT_DIR / "experiment_metadata.json").write_text(json.dumps(metadata, indent=2))
progress("complete", completed_checks=len(CHECKS), full_models=len(TREE_COUNTS))
display(final_training)
print("Completed fixed comparison. No automatic further tuning, final selection or Kaggle upload.")

Saved and verified 150-tree model and 15,329 prediction rows.
Saved and verified 225-tree model and 15,329 prediction rows.
Saved and verified 300-tree model and 15,329 prediction rows.
Completed fixed comparison. No automatic further tuning, final selection or Kaggle upload.


,model,trees,training_rows,training_illicit,test_rows,unknown_training_rows_excluded,saved_model_reload_max_error,submission_sha256,model_sha256,submission
0,lightgbm_150,150,31235,3644,15329,110537,0.0,fb7545a1c1c7269de724d55430217d55ec912ff86fa5db...,2ff9771b75ab03f1260d36815851dfc2022e07c2afaee3...,/Users/mlvlo/.codex/.chatgpt-projects/g-p-6a90...
1,lightgbm_225,225,31235,3644,15329,110537,0.0,c4290e798ec02f215db11542a398345c1a190ee15326f4...,df33e660891e67d93282f19378ebaea5a273dcf49f2eee...,/Users/mlvlo/.codex/.chatgpt-projects/g-p-6a90...
2,lightgbm_300,300,31235,3644,15329,110537,0.0,228f3e14f2edd5e8287bedecce11c7a9120fbb77697f62...,cfee970923707720b96ca9940c5975d1ae09b42f4af4b2...,/Users/mlvlo/.codex/.chatgpt-projects/g-p-6a90...


## Measured findings

The fixed comparison completed all **15 chronological checks**, followed by full fits of the three declared candidates using **31,235 confirmed labels** through step 35. Each candidate generated **15,329 correctly aligned test predictions**. The **110,537 unknown training rows** received no labels and were excluded from these supervised fits. Elapsed experiment time was **53.1 seconds** on CPU.

| Development check group | Submitted ensemble recipe | LightGBM 150 | LightGBM 225 | LightGBM 300 |
|---|---:|---:|---:|---:|
| All 15 | 0.954972 | 0.955732 | 0.952160 | 0.950552 |
| Earlier eight | 0.980638 | 0.979047 | 0.978620 | 0.977470 |
| Later seven | 0.925638 | 0.929086 | 0.921921 | 0.919788 |
| Previously chosen four | 0.935353 | 0.937024 | 0.933004 | 0.932575 |
| Latest: train 1–21, check 22–35 | 0.916956 | 0.919396 | 0.913854 | 0.915263 |

- **225 trees:** wins 4 of 15 and 0 of seven later checks against 150 trees; mean later change -0.007165 AUC.
- **300 trees:** wins 3 of 15 and 0 of seven later checks against 150 trees; mean later change -0.009298 AUC.

**Interpretation:** The 150-tree version has the highest mean among these three candidates on both all 15 and the seven later checks. The longer training variants did not improve those means. This supports keeping 150 trees as the reference for the next student-chosen experiment. The existing 150-tree LightGBM beats the ensemble on **7 of seven later checks**. This is evidence for evaluating its standalone predictions; its actual public score remains unknown. The submitted ensemble's **0.94701** is the only public score attached to that recipe in this comparison.

Extra trees improve training fit by making further corrections, but that does not ensure better ranking of later transactions. The measured decline or improvement applies to these fixed settings and checks; it does not diagnose every source of distribution change. No setting was revised after examining checking scores.

The later checks are more relevant to forecasting future transactions because they contain more recent periods. Earlier checks still reveal tradeoffs. Reusing the same chronological checks provides a fair comparison against the control, but the checks overlap: **163,794 row appearances cover 21,682 unique transactions**, with up to **14 appearances each**. Neither the mean nor the win count is independent held-out proof. No calibrated prediction of Kaggle ROC-AUC, or guarantee of exceeding 0.96, follows from these results.

### Verification and reproducibility

All five code cells executed with genuine captured outputs. Independent recalculation of every AUC, checking identifiers, labels and fitting counts passed. The freshly fitted control differs from the prior control by at most **0 AUC** and the 150-tree LightGBM by at most **0 AUC**. Saved LightGBM text models reproduced full test predictions to within 1e−15 after reload. Submission columns, identifiers, row order, probability range, CSV round trips and source-file hashes passed. The notebook is standalone and requires no prior experiment arrays to generate predictions.

Complete source and executed outputs: `e1156285_lightgbm_length.ipynb`. Tables, saved checking probabilities, three fitted models, experimental prediction CSVs, declared recipe and independent verification: `lightgbm_length_results/`.

**Decision ownership:** the student selected this fixed 150/225/300-tree comparison. OpenAI Codex implemented and ran it, verified results and drafted this record. These are AI-assisted development findings for student review. No autonomous next experiment, automatic blend search, final model selection or Kaggle upload occurred. The student must disclose this assistance accurately and be able to explain the model and validation design.

### All chronological results

| Train through | Check steps | Ensemble | LightGBM 150 | LightGBM 225 | LightGBM 300 |
|---:|---|---:|---:|---:|---:|
| 7 | 8–21 | 0.992985 | 0.991691 | 0.992252 | 0.992588 |
| 8 | 9–22 | 0.993827 | 0.993200 | 0.993783 | 0.993993 |
| 9 | 10–23 | 0.991915 | 0.990934 | 0.991591 | 0.991521 |
| 10 | 11–24 | 0.977749 | 0.976330 | 0.975135 | 0.974008 |
| 11 | 12–25 | 0.975593 | 0.973871 | 0.971075 | 0.967252 |
| 12 | 13–26 | 0.969104 | 0.968787 | 0.966885 | 0.964336 |
| 13 | 14–27 | 0.971758 | 0.969207 | 0.968999 | 0.969035 |
| 14 | 15–28 | 0.972177 | 0.968360 | 0.969240 | 0.967026 |
| 15 | 16–29 | 0.939024 | 0.943194 | 0.939887 | 0.936142 |
| 16 | 17–30 | 0.947183 | 0.948616 | 0.944501 | 0.939395 |
| 17 | 18–31 | 0.939401 | 0.941784 | 0.936758 | 0.935291 |
| 18 | 19–32 | 0.916818 | 0.923267 | 0.912162 | 0.911618 |
| 19 | 20–33 | 0.912879 | 0.918555 | 0.912165 | 0.912719 |
| 20 | 21–34 | 0.907205 | 0.908792 | 0.894119 | 0.888088 |
| 21 | 22–35 | 0.916956 | 0.919396 | 0.913854 | 0.915263 |
